In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [4]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([5], device='cuda:0')


In [7]:
input_image = torch.rand(3,28,28)
print(input_image.size())

flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)

torch.Size([3, 28, 28])
torch.Size([3, 784])
torch.Size([3, 20])
Before ReLU: tensor([[ 0.1019,  0.4952,  0.3666,  0.0758,  0.5858,  0.3178,  0.0619, -0.7891,
         -0.0474, -0.0026, -0.2771,  0.0717, -0.0497,  0.1215,  0.5112,  0.9159,
         -0.1754,  0.7194, -0.3252,  0.3658],
        [-0.2382,  0.3639, -0.2543,  0.2301,  0.1381,  0.3119,  0.1590, -0.6905,
         -0.2947, -0.2105, -0.4439, -0.0217,  0.2164,  0.0030,  0.5641,  1.2601,
          0.2068,  0.1992, -0.2984,  0.0600],
        [ 0.2408,  0.3336,  0.1459,  0.1514,  0.3010,  0.3707,  0.0260, -0.8696,
         -0.0639, -0.0149, -0.2909, -0.3092, -0.0848,  0.2113,  0.2805,  1.2351,
         -0.2439,  0.4971,  0.0627, -0.0059]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.1019, 0.4952, 0.3666, 0.0758, 0.5858, 0.3178, 0.0619, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0717, 0.0000, 0.1215, 0.5112, 0.9159, 0.0000, 0.7194,
         0.0000, 0.3658],
        [0.0000, 0.3639, 0.0000, 0.2301, 0.1381, 0.3119, 0.1590, 0.00

In [8]:
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)
print(logits)

softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
print(pred_probab)

tensor([[-0.0095,  0.0796,  0.3296,  0.1893, -0.0920,  0.0362,  0.2183,  0.0062,
         -0.1719,  0.2711],
        [-0.0123,  0.0234,  0.1470,  0.2171, -0.0137,  0.1214,  0.2224,  0.0807,
         -0.0282,  0.2315],
        [-0.0989,  0.0978,  0.1769,  0.1841, -0.0114,  0.0268,  0.3104, -0.0365,
         -0.0692,  0.2295]], grad_fn=<AddmmBackward0>)
tensor([[0.0898, 0.0982, 0.1261, 0.1096, 0.0827, 0.0940, 0.1128, 0.0913, 0.0764,
         0.1190],
        [0.0890, 0.0923, 0.1044, 0.1120, 0.0889, 0.1018, 0.1126, 0.0977, 0.0876,
         0.1136],
        [0.0828, 0.1008, 0.1091, 0.1099, 0.0904, 0.0939, 0.1247, 0.0881, 0.0853,
         0.1150]], grad_fn=<SoftmaxBackward0>)


In [9]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 3.3335e-02, -2.9984e-02, -5.7974e-03,  ...,  1.2623e-02,
         -1.1925e-02, -3.4548e-02],
        [-2.6243e-05, -2.9936e-02, -8.8893e-03,  ..., -2.2095e-02,
         -1.1323e-02,  2.0417e-02]], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([ 0.0176, -0.0145], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 3.7171e-02,  2.8553e-02, -3.0149e-02,  ..., -2.7776e-02,
          1.3534e-02, -3.6263e-02],
        [-3.37